# Senate Profile LLM Extraction Pipeline
**DSBA 6010 — Chloe Partridge**

Aligned with Liu et al. (USENIX Security 2025) *Evaluating LLM-based Personal Information Extraction and Countermeasures*.

Key features:
- **Prompt-style ablation** — direct, pseudocode, ICL (Section 4.2 / Table 13)
- **Religion signal annotation** — LLM-based pre-classification of bio text as `explicit` / `not_explicit` (Section 8b)
- **Traditional baselines** — keyword search, regex, spaCy NER (Tables 4–5)
- **Evaluation metrics** — Accuracy, Rouge-1, BERT score with stratified religion analysis (Section 6.1.4)
- **Model comparison** — 8B vs 70B (Table 3 / Section 6.2)

In [1]:
# Install required packages for all supported providers
# !pip install beautifulsoup4 google-generativeai openai groq pandas tqdm rouge-score bert-score spacy
# !python -m spacy download en_core_web_sm

## 2. Configuration & Session Setup

###  API Configuration & Session tracking
Initializes API client, sets temperature and token limits, and prepares session metadata.

In [2]:
import warnings
warnings.filterwarnings("ignore", category=UserWarning)

In [3]:
import sys
sys.path.insert(0, '..')

import warnings
warnings.filterwarnings("ignore", category=UserWarning)

import json, time, re, random, os, datetime
from dataclasses import dataclass, asdict
import pandas as pd
from pathlib import Path
from tqdm.notebook import tqdm
from groq import Groq
import spacy
from rouge_score import rouge_scorer
import bert_score

# Module imports
from modules.config import TASK1_DIRECT, TASK1_PSEUDOCODE, TASK1_ICL, T1_FIELDS, REGEX_PATTERNS
from modules.api import call_groq, run_pipeline
from modules.name_utils import NameNormalizer
from modules.html_processing import extract_readable_text
from modules.parsing import parse_education, parse_committee_roles, normalize_degree
from modules.baselines import regex_extract, spacy_extract, keyword_extract
from modules.evaluator import name_match_score, religion_match_score, gender_match_score

# ════════════════════════════════════════════════════════════════════════════
# CENTRALIZED SESSION CONFIGURATION
# ════════════════════════════════════════════════════════════════════════════

@dataclass
class SessionConfig:
    """Unified configuration for the extraction pipeline."""
    # Paths
    output_dir: Path
    html_dir: Path
    config_file: Path
    
    # Model parameters
    model: str
    temperature: float
    max_tokens: int
    api_key: str
    
    # Prompt styles
    prompt_styles: list  # e.g., ["direct", "pseudocode", "icl"]
    prompt_map: dict    # e.g., {"direct": TASK1_DIRECT, ...}
    
    # Session tracking
    timestamp: str
    
    # Derived API client (lazy-initialized)
    api_client: Groq = None
    
    def __post_init__(self):
        """Initialize API client after dataclass creation."""
        if self.api_client is None:
            self.api_client = Groq(api_key=self.api_key)
    
    def to_dict(self):
        """Return serializable config dict (excluding api_client and prompt_map)."""
        return {
            "output_dir": str(self.output_dir),
            "html_dir": str(self.html_dir),
            "config_file": str(self.config_file),
            "model": self.model,
            "temperature": self.temperature,
            "max_tokens": self.max_tokens,
            "prompt_styles": self.prompt_styles,
            "timestamp": self.timestamp
        }


# ── Load model configuration from JSON ─────────────────────────────────────
config_file_path = Path("../configs/model_configs/groq_config_extraction.json")
with open(config_file_path) as f:
    config_json = json.load(f)

api_key = os.getenv("GROQ_API_KEY") or config_json["api_key_info"]["api_keys"][0]
model = config_json["model_info"]["name"]
temp = config_json["params"]["temperature"]
max_tok = config_json["params"]["max_output_tokens"]

# ── Setup paths and initialize spacy ───────────────────────────────────────
output_dir = Path("../outputs/senate_results")
html_dir = Path("../external_data/senate_html")
output_dir.mkdir(parents=True, exist_ok=True)

html_files = sorted(html_dir.glob("*.html"))
nlp = spacy.load('en_core_web_sm')

# ── Create unified session config ──────────────────────────────────────────
# Choose prompt styles here: ["direct"], ["direct", "pseudocode", "icl"], etc.
PROMPT_STYLES = ["direct", "pseudocode", "icl"]

PROMPT_MAP = {
    "direct": TASK1_DIRECT,
    "pseudocode": TASK1_PSEUDOCODE,
    "icl": TASK1_ICL
}

# Validate selected styles
for style in PROMPT_STYLES:
    if style not in PROMPT_MAP:
        raise ValueError(f"Invalid prompt style: {style}. Must be one of {list(PROMPT_MAP.keys())}")

session_config = SessionConfig(
    output_dir=output_dir,
    html_dir=html_dir,
    config_file=config_file_path,
    model=model,
    temperature=temp,
    max_tokens=max_tok,
    api_key=api_key,
    prompt_styles=PROMPT_STYLES,
    prompt_map=PROMPT_MAP,
    timestamp=datetime.datetime.now().isoformat()
)

# ── Convenience aliases for backward compatibility ────────────────────────
OUTPUT_DIR = session_config.output_dir
HTML_DIR = session_config.html_dir
api_client = session_config.api_client

print(f"✓ Model: {session_config.model} | {len(html_files)} HTML files")
print(f"✓ Prompt styles: {', '.join(session_config.prompt_styles)}")
print(f"✓ Output dir: {session_config.output_dir}")

/Users/chloe/LLM-Based-Personal-Profile-Extraction/pie_env/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


✓ Model: llama-3.1-8b-instant | 100 HTML files
✓ Prompt styles: direct, pseudocode, icl
✓ Output dir: ../outputs/senate_results


In [4]:
# Load senator index with URL preparation
senators = pd.read_csv(Path("../external_data/ground_truth/senate_ground_truth.csv"))
senators["wikipedia_url"] = senators["name"].apply(NameNormalizer.create_wikipedia_url)
senators["ballotpedia_url"] = senators["name"].apply(NameNormalizer.create_ballotpedia_url)
print(f"✓ Loaded {len(senators)} senators")

✓ Loaded 99 senators


In [5]:
# Test HTML preprocessing (already imported from modules)
sample = html_files[0]
text = extract_readable_text(sample.read_text(encoding="utf-8", errors="ignore"))
print(f"Sample: {sample.name}")
print(f"Raw: {sample.stat().st_size:,} bytes → Cleaned: {len(text):,} chars")
print(f"First 300 chars: {text[:300]}...")

Sample: Adam_Schiff_CA.html
Raw: 222,779 bytes → Cleaned: 7,918 chars
First 300 chars: About - Senator Schiff Skip to content Home Search Mobile Nav Open Open Search Close Search Contact Adam Close Mobile Nav Search About Senator Adam Schiff About Senator Adam Schiff After Adam graduated from law school, he moved to Los Angeles to serve as a law clerk for Judge William Matthew Byrne, ...


In [6]:
# Display session configuration (all settings consolidated in session_config)
print("=" * 70)
print("SESSION CONFIGURATION")
print("=" * 70)
config_dict = session_config.to_dict()
for key, value in config_dict.items():
    if key == "prompt_styles":
        print(f"  {key:.<30s} {', '.join(value)}")
    else:
        print(f"  {key:.<30s} {value}")
print("=" * 70)

SESSION CONFIGURATION
  output_dir.................... ../outputs/senate_results
  html_dir...................... ../external_data/senate_html
  config_file................... ../configs/model_configs/groq_config_extraction.json
  model......................... llama-3.1-8b-instant
  temperature................... 0.1
  max_tokens.................... 1500
  prompt_styles................. direct, pseudocode, icl
  timestamp..................... 2026-04-08T21:16:53.275021


In [7]:
# ════════════════════════════════════════════════════════════════════════════
# API INFRASTRUCTURE - call_groq and run_pipeline
# ════════════════════════════════════════════════════════════════════════════
#
# These core functions are now imported from modules/api.py:
#
#   - call_groq()    : Call Groq API, parse JSON response, exponential backoff
#   - run_pipeline() : Orchestrate all prompt styles, return structured results
#
# Location: modules/api.py
# See above in cell 2 for import statement
# ════════════════════════════════════════════════════════════════════════════

print("✓ call_groq and run_pipeline imported from modules.api")


✓ call_groq and run_pipeline imported from modules.api


## 5. API Infrastructure

Helper functions for LLM API calls with automatic retry on rate-limit errors.
Includes exponential backoff, JSON parsing, and model-agnostic abstraction layer.

In [8]:
# Safety check: re-initialize session_config and html_files if kernel was restarted
if 'session_config' not in dir() or 'html_files' not in dir():
    print("  Re-initializing after kernel restart...")
    from modules import PATHS, extract_readable_text
    session_config = PATHS['output_root']
    html_files = sorted(session_config.html_dir.glob("*.html"))
    print(f"✓ Reinitialized: {len(html_files)} HTML files found")

raw_path = session_config.output_dir / "results_raw.json"

if raw_path.exists():
    with open(raw_path) as f:
        results = json.load(f)
    done_ids = {r["senator_id"] for r in results}
    print(f"Resuming — {len(done_ids)} senators already processed.")
else:
    results, done_ids = [], set()

remaining = [f for f in html_files if f.stem not in done_ids]
print(f"Senators remaining: {len(remaining)}")

# Rate limiting adapted to number of prompt styles
# When running multiple styles, each senator gets N API calls
# With inter-style delays, we need longer main delays to stay under quotas
inter_senator_delay = 6 if len(session_config.prompt_styles) > 1 else 4

print(f"Rate limit: {inter_senator_delay}s between senators")
print(f"Processing with {len(session_config.prompt_styles)} prompt style(s)\n")

for html_file in tqdm(remaining, desc="Processing senators"):
    senator_id = html_file.stem
    html       = html_file.read_text(encoding="utf-8", errors="ignore")
    text       = extract_readable_text(html)
    result     = run_pipeline(text, session_config)

    results.append({"senator_id": senator_id, **result})
    with open(raw_path, "w") as f:
        json.dump(results, f, indent=2)

    time.sleep(inter_senator_delay)

print(f"\nDone. {len(results)} senators processed.")

Resuming — 100 senators already processed.
Senators remaining: 0
Rate limit: 6s between senators
Processing with 3 prompt style(s)



Processing senators: 0it [00:00, ?it/s]


Done. 100 senators processed.


In [9]:
with open(OUTPUT_DIR / "results_raw.json") as f:
    results = json.load(f)

# Flatten Task 1 results to CSV
task1_rows = []
for r in results:
    t1_data = r.get("task1_pii", {})
    prompt_style = r.get("prompt_style", "unknown")
    
    # Handle both single-style and all-styles results
    if prompt_style == "all_styles":
        # All three styles were run — create separate rows for each style
        for style_name, style_result in t1_data.items():
            row = {
                "senator_id": r["senator_id"],
                "prompt_style": style_name,
                "extraction_error": style_result.get("error")
            }
            for field in T1_FIELDS:
                row[field] = style_result.get(field)
            task1_rows.append(row)
    else:
        # Single style was run
        row = {
            "senator_id": r["senator_id"],
            "prompt_style": prompt_style,
            "extraction_error": t1_data.get("error")
        }
        for field in T1_FIELDS:
            row[field] = t1_data.get(field)
        task1_rows.append(row)

df_t1 = pd.DataFrame(task1_rows)
df_t1.to_csv(OUTPUT_DIR / "task1_pii.csv", index=False)
print(f"Saved {len(df_t1)} rows to task1_pii.csv")
print(f"\nPrompt styles in results:")
print(df_t1["prompt_style"].value_counts().to_string())

Saved 300 rows to task1_pii.csv

Prompt styles in results:
prompt_style
direct        100
pseudocode    100
icl           100


In [10]:
print("=== Task 1: Field Coverage ===")
for col in ["full_name","birthdate","birth_year_inferred","gender","race_ethnicity","education","committee_roles","religious_affiliation","religious_affiliation_inferred"]:
    filled = df_t1[col].notna().sum()
    print(f"  {col:40s}: {filled}/{len(df_t1)}")

=== Task 1: Field Coverage ===
  full_name                               : 300/300
  birthdate                               : 51/300
  birth_year_inferred                     : 0/300
  gender                                  : 267/300
  race_ethnicity                          : 17/300
  education                               : 300/300
  committee_roles                         : 299/300
  religious_affiliation                   : 97/300
  religious_affiliation_inferred          : 300/300


## 8. Baseline Methods (Liu et al. Section 5, Tables 4–5)

Implements traditional information extraction methods for comparison:
- **Regex baseline** — Rule-based patterns for name, email, phone
- **spaCy NER baseline** — Pre-trained neural entity recognition for PERSON, ORG, GEO

Purpose: Quantify LLM superiority on canonical PII extraction tasks. 
This section tests both methods on the same HTML data then compares coverage vs. LLM accuracy.

In [11]:
# ════════════════════════════════════════════════════════════════════════════
# SECTION 7: GROUND TRUTH COLLECTION (Optional — for evaluation)
# ════════════════════════════════════════════════════════════════════════════
# 
# What we've done so far:
#   ✓ Set up API configuration (Section 2)
#   ✓ Run Task 1 extraction on all HTML files (Section 6)
#   ✓ Built baseline extractors — regex + spaCy (Section 8)
#
# What we're doing now:
#   - Scrape Wikipedia & Ballotpedia for ground truth labels
#   - Build comprehensive gold-standard dataset for evaluation
#   - Cache Wikipedia text for later education extraction (Section 7b)
#
# Output:
#   - senate_ground_truth.csv with all gold-standard labels
#   - WIKI_CACHE dict with cached biographical texts
#
# Note: This section takes ~2-3 minutes due to web scraping delays.
# Can be skipped if ground truth CSV already exists.
# ════════════════════════════════════════════════════════════════════════════

In [12]:
# ── Setup paths for ground truth collection ─────────────────────────────
# Define output paths for ground truth CSV and logging
OUTPUT_PATH = Path("../external_data/ground_truth/senate_ground_truth.csv")
LOG_PATH = session_config.output_dir / "ground_truth_errors.log"

print(f"✓ Ground truth output: {OUTPUT_PATH}")
print(f"✓ Error log: {LOG_PATH}")

✓ Ground truth output: ../external_data/ground_truth/senate_ground_truth.csv
✓ Error log: ../outputs/senate_results/ground_truth_errors.log


In [13]:
# Check if output file exists — if so, load to avoid re-processing completed records
if Path(OUTPUT_PATH).exists():
    existing_df = pd.read_csv(OUTPUT_PATH)
    # Track which senators are already done (ground truth doesn't have prompt_style variation)
    existing_combos = set(existing_df["senator_id"]) if "senator_id" in existing_df.columns else set()
    print(f"✓ Found {len(existing_df)} existing records — will resume (skip completed)")
else:
    existing_combos = set()


✓ Found 99 existing records — will resume (skip completed)


In [14]:
# ── Merge Pew religion data and print coverage summary ───────────────────
# This cell processes a ground truth CSV if it exists (from scraping phase)
# If the file doesn't exist yet, this cell is optional and can be skipped

if not OUTPUT_PATH.exists():
    print(f"⚠ Ground truth file not yet created: {OUTPUT_PATH}")
    print("  This file would be generated by running the ground truth scraping section.")
    print("  Skipping Pew religion merge for now.\n")
else:
    PEW_PATH = Path("../external_data/pew_religion.csv")
    
    # Load the saved ground truth CSV
    df_gt = pd.read_csv(OUTPUT_PATH)
    
    if "senator_id" not in df_gt.columns:
        df_gt["senator_id"] = df_gt["name"].str.replace(" ", "_") + "_" + df_gt["state"]
    
    # Attempt to add Pew religion data if file exists
    if PEW_PATH.exists():
        try:
            # Try to import and use merge_pew if available
            from groundtruth import merge_pew
            religion_series = merge_pew(df_gt, str(PEW_PATH))
            df_gt["religion"] = religion_series.values
            pew_merged = True
        except (ImportError, Exception) as e:
            print(f"  Note: Could not merge Pew data ({type(e).__name__})")
            pew_merged = False
    else:
        pew_merged = False
    
    # Save ground truth
    df_gt.to_csv(OUTPUT_PATH, index=False)
    
    print("\n" + "="*70)
    print(" GROUND TRUTH COVERAGE SUMMARY")
    print("="*70)
    print(f"Total senators in ground truth: {len(df_gt)}\n")
    
    coverage_fields = ["full_name", "birthdate", "gender", "race_ethnicity",
                       "committee_roles"]
    if "religion" in df_gt.columns:
        coverage_fields.append("religion")
    
    coverage_stats = []
    
    for field in coverage_fields:
        # Count non-null, non-NaN values
        non_null = df_gt[field].notna().sum()
        pct = (non_null / len(df_gt)) * 100 if len(df_gt) > 0 else 0
        coverage_stats.append({
            "Field": field,
            "Non-Null": non_null,
            "Coverage": f"{pct:.1f}%"
        })
    
    df_coverage = pd.DataFrame(coverage_stats)
    print(df_coverage.to_string(index=False))
    print("="*70)
    print(f"\n✓ Final output saved to: {OUTPUT_PATH}")
    print(f"✓ Errors logged to: {LOG_PATH}")
    if pew_merged:
        print(f"✓ Religion data merged from: {PEW_PATH}\n")
    else:
        print(f"  Note: Pew religion data not merged (file not found or import failed)\n")

  Note: Could not merge Pew data (ModuleNotFoundError)

 GROUND TRUTH COVERAGE SUMMARY
Total senators in ground truth: 99

          Field  Non-Null Coverage
      full_name        97    98.0%
      birthdate        94    94.9%
         gender        94    94.9%
 race_ethnicity         0     0.0%
committee_roles        79    79.8%
       religion        96    97.0%

✓ Final output saved to: ../external_data/ground_truth/senate_ground_truth.csv
✓ Errors logged to: ../outputs/senate_results/ground_truth_errors.log
  Note: Pew religion data not merged (file not found or import failed)



In [15]:
# ── Education ground truth extraction from Wikipedia (with caching) ────────
# Uses Wikipedia page text + LLM extraction, saved for manual review.
# WIKI_CACHE is reused from scraping phase to avoid re-fetching.
# After running, manually verify education_text column before using in evaluation.

# Check if ground truth file exists first
if not Path(OUTPUT_PATH).exists():
    print(f"⚠ Ground truth file not yet created: {OUTPUT_PATH}")
    print("  This file would be generated by running the ground truth scraping section.")
    print("  Please run the 'Merge Pew religion data' cell first to create the ground truth file.")
    print("  Skipping education extraction for now.\n")
else:
    # Reload groundtruth module to get fresh functions
    import importlib
    import sys
    if 'modules.groundtruth' in sys.modules:
        del sys.modules['modules.groundtruth']

    from modules.groundtruth import fetch_wikipedia_text
    from modules import NameNormalizer
    from modules.config import EDUCATION_PROMPT
    from tqdm import tqdm

    df_gt = pd.read_csv(OUTPUT_PATH)

    if "education_text" not in df_gt.columns:
        df_gt["education_text"] = None

    needs_education = df_gt[df_gt["education_text"].isna()]
    print(f"Senators needing education extraction: {len(needs_education)}")
    
    # Initialize WIKI_CACHE if it doesn't exist (e.g., if ground truth scraping wasn't run)
    if 'WIKI_CACHE' not in dir():
        WIKI_CACHE = {}
    
    print(f"Using WIKI_CACHE: {len(WIKI_CACHE)} cached pages\n")

    cache_hits = 0
    cache_misses = 0

    for idx, row in tqdm(needs_education.iterrows(), total=len(needs_education),
                         desc="Extracting education"):
        senator_name = row["name"]
        wiki_url = NameNormalizer.create_wikipedia_url(senator_name)

        # Try to use cached text first (from scraping phase)
        if senator_name in WIKI_CACHE:
            wiki_text = WIKI_CACHE[senator_name]
            cache_hits += 1
            
            # Skip if fetch failed previously (cached as "error")
            if wiki_text == "error":
                df_gt.at[idx, "education_text"] = "error"
                df_gt.to_csv(OUTPUT_PATH, index=False)
                continue
        else:
            # Cache miss — fetch now
            cache_misses += 1
            wiki_text = fetch_wikipedia_text(wiki_url, senator_name)
            WIKI_CACHE[senator_name] = wiki_text
            
            if wiki_text == "error":
                df_gt.at[idx, "education_text"] = "error"
                df_gt.to_csv(OUTPUT_PATH, index=False)
                time.sleep(1)
                continue
            
            time.sleep(1.5)  # Be respectful with cache misses

        # Extract education using LLM on cached text
        result = call_groq(EDUCATION_PROMPT, wiki_text, config=session_config, max_retries=3)

        if isinstance(result, dict):
            raw = result.get("raw", result.get("error", ""))
            df_gt.at[idx, "education_text"] = raw.strip() if raw else "error"
        else:
            df_gt.at[idx, "education_text"] = str(result).strip()

        df_gt.to_csv(OUTPUT_PATH, index=False)
        time.sleep(0.5)

    print(f"\n{'='*70}")
    print("✓ Education extraction complete")
    print(f"  Cache hits: {cache_hits} (no network fetch needed)")
    print(f"  Cache misses: {cache_misses} (fetched from Wikipedia)")
    print(f"  Please manually verify education_text column before using in evaluation")
    print(f"  Saved to: {OUTPUT_PATH}")
    print(f"{'='*70}\n")

Senators needing education extraction: 3
Using WIKI_CACHE: 0 cached pages



Extracting education: 100%|██████████| 3/3 [00:07<00:00,  2.43s/it]


✓ Education extraction complete
  Cache hits: 0 (no network fetch needed)
  Cache misses: 3 (fetched from Wikipedia)
  Please manually verify education_text column before using in evaluation
  Saved to: ../external_data/ground_truth/senate_ground_truth.csv



In [16]:
## ════════════════════════════════════════════════════════════════════════════
## SECTION 9: COMPREHENSIVE EVALUATION
## ════════════════════════════════════════════════════════════════════════════
## What we've done so far:
##   1. Extracted PII from all senator profiles using 3 prompt styles (Section 6)
##   2. Built ground truth from Wikipedia/Ballotpedia/Pew data (Section 7)
## 
## What we're doing now:
##   - Compare LLM predictions vs ground truth using task-specific metrics
##   - Compute accuracy per field with hierarchical matching where appropriate
##   - Stratify results by prompt style to show effect of prompt engineering
## 
## Output:
##   - Accuracy metrics printed per style (direct, pseudocode, ICL)
##   - Use for paper Tables 6-9 and ablation discussion
## ════════════════════════════════════════════════════════════════════════════

# ── Load predictions and ground truth dataframes ────────────────────────
# Load LLM predictions from task1_pii.csv
if not Path(OUTPUT_PATH).exists():
    print(f"⚠ Cannot run evaluation — ground truth file not created yet: {OUTPUT_PATH}")
    print("  Please run the ground truth scraping section first.\n")
else:
    # Load predictions (df_pred) and ground truth (df_gt)
    df_pred = pd.read_csv(OUTPUT_DIR / "task1_pii.csv")
    df_gt = pd.read_csv(OUTPUT_PATH)
    
    print(f"✓ Loaded {len(df_pred)} predictions (all prompt styles)")
    print(f"✓ Loaded {len(df_gt)} ground truth records")
    print(f"✓ Predictions prompt styles: {df_pred['prompt_style'].unique()}")
    print(f"\nReady for evaluation.\n")

✓ Loaded 300 predictions (all prompt styles)
✓ Loaded 99 ground truth records
✓ Predictions prompt styles: ['direct' 'pseudocode' 'icl']

Ready for evaluation.



In [17]:
# ── Define evaluation helper functions ───────────────────────────────────
# Suppress BERTScore model loading warnings
import warnings
warnings.filterwarnings("ignore", message=".*Some weights of RobertaModel.*")

# Import evaluation functions from modules
from modules.evaluator import (
    match_by_fuzzy_name,
    evaluate_text_fields,
    evaluate_education_components
)

# ROUGE scorer setup
scorer_r = rouge_scorer.RougeScorer(['rouge1'], use_stemmer=True)

print("✓ Evaluation functions imported from modules.evaluator")

✓ Evaluation functions imported from modules.evaluator


In [18]:
# ── 9B. Evaluate: Full Name & Gender Accuracy ─────────────────────────────
# For each prompt style (direct, pseudocode, icl), compute accuracy metrics
# by comparing LLM predictions to ground truth

for style in ["direct", "pseudocode", "icl"]:
    print(f"\n{'='*70}")
    print(f"PROMPT STYLE: {style.upper()}  (Ground Truth ↔ LLM Predictions)")
    print(f"{'='*70}\n")
    
    # Filter predictions to this style
    df_style = df_pred[df_pred["prompt_style"] == style]

    # TRY 1: Merge by exact senator_id match (fast path)
    merged = df_gt.merge(df_style, on="senator_id", how="inner")

    # TRY 2: Fallback to fuzzy name matching if no exact matches found
    # This handles cases where senator IDs don't match but names do
    if merged.empty and len(df_style) > 0:
        print(f"  No direct senator_id matches — falling back to fuzzy name matching\n")
        merged = match_by_fuzzy_name(df_gt, df_style)

    # Skip this style if still no matches
    if merged.empty:
        print(f"✗ No matches found for {style} — skipping\n")
        continue

    print(f"Matched records: {len(merged)}/{len(df_gt)} senators\n")


PROMPT STYLE: DIRECT  (Ground Truth ↔ LLM Predictions)

Matched records: 99/99 senators


PROMPT STYLE: PSEUDOCODE  (Ground Truth ↔ LLM Predictions)

Matched records: 99/99 senators


PROMPT STYLE: ICL  (Ground Truth ↔ LLM Predictions)

Matched records: 99/99 senators



In [19]:
# ── 9C.5 Evaluate: Gender Accuracy (Exact Match) ──────────────────────────

for style in ["direct", "pseudocode", "icl"]:
    df_style = df_pred[df_pred["prompt_style"] == style]
    merged = df_gt.merge(df_style, on="senator_id", how="inner")
    
    if merged.empty:
        merged = match_by_fuzzy_name(df_gt, df_style)
    if merged.empty:
        continue

    print(f"\n[{style.upper()}]")
    
    # ── Gender (exact match, case-insensitive) ──────────────────
    # Find gender columns (may be suffixed after merge if duplicate names exist)
    gt_gender_col = next((c for c in ["gender_x", "gender"] 
                          if c in merged.columns), None)
    pred_gender_col = next((c for c in ["gender_y", "gender"] 
                            if c in merged.columns), None)

    if gt_gender_col and pred_gender_col:
        gender_scores = [
            gender_match_score(gt, pred)
            for gt, pred in zip(merged[gt_gender_col], merged[pred_gender_col])
        ]
        
        valid = [s for s in gender_scores if not pd.isna(s)]
        if valid:
            acc = sum(valid) / len(valid)
            missing_gt = sum(1 for s in gender_scores if pd.isna(s))
            print(f"   Gender (exact match):     {acc:.2%}  "
                  f"(n={len(valid)}, missing_gt={missing_gt})")



[DIRECT]
   Gender (exact match):     100.00%  (n=94, missing_gt=5)

[PSEUDOCODE]
   Gender (exact match):     70.21%  (n=94, missing_gt=5)

[ICL]
   Gender (exact match):     98.94%  (n=94, missing_gt=5)


In [20]:
# ── 9E. Evaluate: Education & Committees (ROUGE, BERT, Components) ──────

for style in ["direct", "pseudocode", "icl"]:
    df_style = df_pred[df_pred["prompt_style"] == style]
    merged = df_gt.merge(df_style, on="senator_id", how="inner")
    
    if merged.empty:
        merged = match_by_fuzzy_name(df_gt, df_style)
    if merged.empty:
        continue

    print(f"\n[{style.upper()}]")

    # ── Text-overlap metrics (ROUGE-1) and BERTScore ──────────────────
    print(f"   Text-Overlap Metrics (ROUGE-1):")
    
    text_fields = [
        ("education",       "education_text",    "education",       parse_education),
        ("committee_roles", "committee_roles_x", "committee_roles_y", parse_committee_roles),
    ]
    
    text_results = evaluate_text_fields(merged, text_fields, scorer_r, bert_scorer=None)
    
    for field_name, metrics in text_results.items():
        print(f"     {field_name:<20}: {metrics['rouge1']:.3f}")

    # ── Semantic similarity (BERTScore) ──────────────────────────
    print(f"   Semantic Similarity (BERT Score - F1):")
    
    text_results = evaluate_text_fields(merged, text_fields, scorer_r, bert_scorer=bert_score)
    
    for field_name, metrics in text_results.items():
        if metrics['bertscore_f1'] is not None:
            print(f"     {field_name:<20}: {metrics['bertscore_f1']:.3f}")

    # ── Education component breakdown ────────────────────────────
    if "education" in merged.columns and "education_text" in merged.columns:
        print(f"   Education Components (Degree / Institution / Year):")
        
        comp_results = evaluate_education_components(merged)
        
        # Print component-level metrics
        for comp_name in ["degree_exact", "institution_fuzzy", "year_exact"]:
            if comp_name in comp_results:
                score = comp_results[comp_name]
                if not pd.isna(score):
                    n = comp_results.get("n", 0)
                    print(f"     {comp_name:<20}: {score:.2%}  (n={n})")
        
        # Print combined score
        combined_score = comp_results.get("combined_score")
        if not pd.isna(combined_score):
            n = comp_results.get("n", 0)
            print(f"     {'combined_all':<20}: {combined_score:.2%}  (n={n})")

    print()  # Spacing between styles



[DIRECT]
   Text-Overlap Metrics (ROUGE-1):
     education           : 0.391
     committee_roles     : 0.270
   Semantic Similarity (BERT Score - F1):


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


     education           : 0.670
     committee_roles     : 0.772
   Education Components (Degree / Institution / Year):
     degree_exact        : 17.75%  (n=74)
     institution_fuzzy   : 65.54%  (n=74)
     year_exact          : 69.57%  (n=74)
     combined_all        : 48.99%  (n=74)


[PSEUDOCODE]
   Text-Overlap Metrics (ROUGE-1):
     education           : 0.384
     committee_roles     : 0.271
   Semantic Similarity (BERT Score - F1):


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


     education           : 0.644
     committee_roles     : 0.703
   Education Components (Degree / Institution / Year):
     degree_exact        : 18.43%  (n=71)
     institution_fuzzy   : 69.84%  (n=71)
     year_exact          : 78.95%  (n=71)
     combined_all        : 51.21%  (n=71)


[ICL]
   Text-Overlap Metrics (ROUGE-1):
     education           : 0.383
     committee_roles     : 0.239
   Semantic Similarity (BERT Score - F1):


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


     education           : 0.636
     committee_roles     : 0.760
   Education Components (Degree / Institution / Year):
     degree_exact        : 44.70%  (n=70)
     institution_fuzzy   : 73.21%  (n=70)
     year_exact          : 75.00%  (n=70)
     combined_all        : 61.71%  (n=70)



In [21]:
# ── Run baselines (regex + spaCy) on all processed profiles ────────────────
# This replicates Liu et al. Tables 4–5: compares LLM extraction vs traditional NER methods
# on the same HTML data, showing LLM's advantages

# Import convenience functions (no class instantiation needed)
from modules import regex_extract, spacy_extract

baseline_rows = []
first_iteration = True

# Iterate over all HTML files processed by the main pipeline
for hf in tqdm(html_files, desc="Baselines"):
    html = hf.read_text(encoding="utf-8", errors="ignore")
    # Use same cleaning as main pipeline for fair comparison
    text = extract_readable_text(html)
    
    # Extract using regex baseline (pattern matching for email, phone, names)
    regex_r = regex_extract(text, REGEX_PATTERNS)
    # Extract using spaCy NER baseline (entity recognition)
    spacy_r = spacy_extract(text, nlp)
    
    # Validation log on first iteration (replaces separate smoke test)
    if first_iteration:
        print(f"✓ Baseline validation on: {hf.name}")
        print(f"  Regex result: {regex_r}")
        print(f"  spaCy result: {spacy_r}\n")
        first_iteration = False
    
    # Flatten results into one row
    baseline_rows.append({
        "senator_id":             hf.stem,
        "regex_name":             regex_r["full_name"],
        "regex_email_found":      1 if regex_r["email"] else 0,
        "regex_phone_found":      1 if regex_r["phone"] else 0,
        "spacy_top_person":       spacy_r["persons_detected"][0] if spacy_r["persons_detected"] else None,
        "spacy_orgs_count":       len(spacy_r["orgs_detected"]),
    })

# Save baseline results to CSV
df_bl = pd.DataFrame(baseline_rows)
df_bl.to_csv(OUTPUT_DIR / "baselines.csv", index=False)

# ── Compare coverage: LLM vs baselines ────────────────────────────────────
# Merge LLM results with baseline results to compare directly
df_t1  = pd.read_csv(OUTPUT_DIR / "task1_pii.csv")
merged_bl = df_t1.merge(df_bl, on="senator_id")

# Calculate coverage percentages (what fraction of records have non-null values)
llm_name_cov   = merged_bl["full_name"].notna().mean()
regex_name_cov = merged_bl["regex_name"].notna().mean()
spacy_name_cov = merged_bl["spacy_top_person"].notna().mean()

llm_relig_cov   = merged_bl["religious_affiliation"].notna().mean()

# Print comparison table (shows LLM superiority on most fields)
print("=== Name extraction coverage (Liu et al. Table 4–5 analog) ===")
print(f"  LLM:   {llm_name_cov:.1%}")
print(f"  Regex: {regex_name_cov:.1%}")
print(f"  spaCy: {spacy_name_cov:.1%}")
print("\n=== Religious Affiliation extraction coverage ===")
print(f"  LLM:   {llm_relig_cov:.1%}")
print("\nFull baseline results saved to baselines.csv")

Baselines:   1%|          | 1/100 [00:00<00:23,  4.24it/s]

✓ Baseline validation on: Adam_Schiff_CA.html
  Regex result: {'full_name': 'Schiff Skip', 'party': 'Democratic', 'email': [], 'phone': [], 'years_found': ['1996', '2000', '2000', '2024', '2019']}
  spaCy result: {'persons_detected': ['Donald J. Trump', 'Instagram Youtube', 'Adam', 'Edward', 'Sherrill Ann Schiff'], 'orgs_detected': ['the Senate Judiciary Committee', 'Danville’s Monte Vista High School', 'Apollo Project-level', 'the House Select Committee', 'FBI', 'Azusa', 'Home Search Mobile Nav Open Open Search', 'Senate', 'the Senate Select Committee', 'Medi-Cal'], 'gpe_detected': ['America', 'Ukraine', 'San Gabriel', 'U.S.', 'Alamo'], 'dates_detected': ['2019', 'the\xa0', 'Early Days Adam', 'four-year', '2000']}



Baselines: 100%|██████████| 100/100 [00:09<00:00, 10.52it/s]

=== Name extraction coverage (Liu et al. Table 4–5 analog) ===
  LLM:   100.0%
  Regex: 99.0%
  spaCy: 100.0%

=== Religious Affiliation extraction coverage ===
  LLM:   30.3%

Full baseline results saved to baselines.csv


In [22]:
# ── Run KeywordSearchBaseline on sample profiles ─────────────────────────
# Tests keyword-based extraction (Liu et al. baseline method)
# More sophisticated than regex, handles HTML structure

from modules import keyword_extract, DEFAULT_KEYWORD_MAP
import warnings
warnings.filterwarnings('ignore')

print("\n" + "="*70)
print(" KEYWORD SEARCH BASELINE")
print("="*70)
print("\nTesting on first 10 profiles...\n")

keyword_rows = []
sample_files = html_files[:10]  # Test on sample for speed

for hf in tqdm(sample_files, desc="Keyword baseline", ncols=70):
    html = hf.read_text(encoding="utf-8", errors="ignore")
    text = extract_readable_text(html)
    
    # Extract using keyword search baseline 
    kw_result = keyword_extract(html, DEFAULT_KEYWORD_MAP)  # Pass raw HTML for better results
    
    # Flatten results into one row
    keyword_rows.append({
        "senator_id": hf.stem,
        "kw_name": kw_result.get("name"),
        "kw_email": kw_result.get("email"),
        "kw_phone": kw_result.get("phone"),
        "kw_state": kw_result.get("state"),
    })

df_keywords = pd.DataFrame(keyword_rows)
print(f"✓ Keyword baseline completed on {len(df_keywords)} profiles")
print(f"\nSample results:")
print(df_keywords.head(3).to_string())
print(f"\nField coverage (keyword baseline):")
for col in df_keywords.columns:
    if col != "senator_id":
        filled = df_keywords[col].notna().sum()
        pct = (filled / len(df_keywords)) * 100
        print(f"  {col:<20}: {filled}/{len(df_keywords)} ({pct:.1f}%)")



 KEYWORD SEARCH BASELINE

Testing on first 10 profiles...



Keyword baseline: 100%|███████████████| 10/10 [00:01<00:00,  8.57it/s]

✓ Keyword baseline completed on 10 profiles

Sample results:
          senator_id kw_name            kw_email kw_phone                                                                kw_state
0     Adam_Schiff_CA    None                None     None                                                                    None
1  Alan_Armstrong_OK    None  Share Your Opinion     None  State Federal Building Street Address City, State, Zip P# 202-222-2222
2    Alex_Padilla_CA    None             Español     None                                                                        

Field coverage (keyword baseline):
  kw_name             : 1/10 (10.0%)
  kw_email            : 8/10 (80.0%)
  kw_phone            : 0/10 (0.0%)
  kw_state            : 3/10 (30.0%)


## 10. Prompt Ablation Study (Liu et al. Section 4.2, Table 13)

Rigorous comparison of all three prompt styles on a **fixed held-out subset of 25 senators**.
Each senator is extracted independently using direct, pseudocode, and ICL prompts.

**Purpose:** Quantify which fields benefit most from structured prompts (e.g., pseudocode) vs. demonstrations (e.g., ICL).
This replicates Liu et al.'s ablation methodology (Section 4.2, Table 13).

**Reproducibility:**
- Fixed random seed (42) ensures same 25 senators across runs
- Separate results file (`ablation_results.json`) keeps ablation independent from main pipeline
- Resume-safe — skips already-completed combinations
- 3-second rate limit between API calls to respect quota

**Expected outcome:** ICL shows largest gains on education, committee roles, and religious affiliation (Liu et al. reports ~15-25% improvement)

In [23]:
# ── Define all prompt styles for ablation ───────────────────────────────
# These are the same prompts as the main pipeline, but applied systematically
# to a fixed subset for controlled comparison
ABLATION_STYLES = {
    "direct": TASK1_DIRECT,
    "pseudocode": TASK1_PSEUDOCODE,
    "icl": TASK1_ICL
}

# ── Load or initialize ablation results ──────────────────────────────────
# Resume-safe: if ablation was interrupted, load progress and continue
ablation_path = OUTPUT_DIR / "ablation_results.json"

if ablation_path.exists():
    with open(ablation_path) as f:
        ablation_results = json.load(f)
    # Track which (senator_id, style) combos are already completed
    completed = set()
    for sid, styles_dict in ablation_results.items():
        for style in styles_dict.keys():
            completed.add((sid, style))
    print(f"✓ Resuming — {len(completed)} senator-style combinations already processed")
else:
    ablation_results = {}
    completed = set()

# ── Compute remaining tasks ──────────────────────────────────────────────
# Each senator needs to be extracted with all 3 styles
ablation_tasks = [(sid, style) for sid in ablation_senators for style in ABLATION_STYLES.keys()]
remaining_tasks = [t for t in ablation_tasks if t not in completed]

print(f"Remaining ablation tasks: {len(remaining_tasks)}/{len(ablation_tasks)}")
print()

# Main ablation loop: extract each senator with each prompt style
for senator_id, style_name in tqdm(remaining_tasks, desc="Ablation loop"):
    # Load HTML and extract clean text (same preprocessing as main pipeline)
    html_file = [f for f in html_files if f.stem == senator_id]
    if not html_file:
        continue
    
    # Read and clean HTML
    html = html_file[0].read_text(encoding="utf-8", errors="ignore")
    text = extract_readable_text(html)
    
    # Extract with current style using Groq API
    prompt = ABLATION_STYLES[style_name]
    result = call_groq(prompt, text)
    
    # Store result in nested dict: {senator_id: {style_name: result, ...}, ...}
    if senator_id not in ablation_results:
        ablation_results[senator_id] = {}
    
    ablation_results[senator_id][style_name] = result
    
    # Save incrementally (critical for resume safety and data preservation)
    with open(ablation_path, "w") as f:
        json.dump(ablation_results, f, indent=2)
    
    # Rate limit protection — 3 seconds between API calls to avoid quota errors
    time.sleep(3)

print(f"\n✓ Ablation complete. Results saved to: ablation_results.json")

✓ Resuming — 24 senator-style combinations already processed


NameError: name 'ablation_senators' is not defined

In [ ]:
# ── Coverage comparison: field coverage per prompt style ─────────────────
print("\n" + "=" * 80)
print(" PROMPT ABLATION — FIELD COVERAGE COMPARISON")
print("=" * 80)
print(f"\nAblation subset size: {len(ablation_senators)} senators")
print(f"Ablation styles: {', '.join(ABLATION_STYLES.keys())}")
print()

# Calculate coverage for each field by prompt style
coverage_by_style = {}
for style in ABLATION_STYLES.keys():
    style_data = df_ablation[df_ablation["prompt_style"] == style].copy()

    error_mask = style_data.get("extraction_error", pd.Series(dtype=str)).notna()
    valid_data = style_data[~error_mask]
    n_errors = int(error_mask.sum())
    n_valid = len(valid_data)

    if n_valid == 0:
        print(f"  ⚠ {style}: all {n_errors} results are errors — delete ablation_results.json and rerun")
        continue

    coverage_by_style[style] = {}
    for field in T1_FIELDS_ABLATION:
        filled = int(valid_data[field].notna().sum())
        pct = (filled / n_valid * 100)
        coverage_by_style[style][field] = {"filled": filled, "total": n_valid, "pct": pct}

if not coverage_by_style:
    print("⚠ No valid results to display.")
else:
    # Print coverage table
    print(f"{'Field':<35} | {'DIRECT':^15} | {'PSEUDOCODE':^15} | {'ICL':^15}")
    print("-" * 80)
    for field in T1_FIELDS_ABLATION:
        row_str = f"{field:<35} | "
        for style in ["direct", "pseudocode", "icl"]:
            if style in coverage_by_style:
                c = coverage_by_style[style][field]
                row_str += f"{c['filled']}/{c['total']} ({c['pct']:5.1f}%) | "
            else:
                row_str += f"{'N/A':^15} | "
        print(row_str)
    print("=" * 80)

    # Show average coverage per style
    print("\n OVERALL COVERAGE BY PROMPT STYLE:")
    for style in ["direct", "pseudocode", "icl"]:
        if style in coverage_by_style:
            avg_cov = sum(coverage_by_style[style][f]["pct"] for f in T1_FIELDS_ABLATION) / len(T1_FIELDS_ABLATION)
            print(f"  {style:<15}: {avg_cov:6.1f}% (avg across all fields)")
        else:
            print(f"  {style:<15}: N/A (all errors)")

    print("\n KEY OBSERVATIONS:")
    print("  • Compare direct vs pseudocode: Liu et al. find pseudocode slightly better for structured fields")
    print("  • Compare direct/pseudocode vs ICL: Liu et al. find ICL best for occupation-type fields")
    print("    (committee_roles, education, religious_affiliation)")
    print("=" * 80 + "\n")

## 11. Appendix: Religion Taxonomy & Hierarchical Matching

To give partial credit for related religions (e.g., Methodist → Christian),
we use a hierarchical taxonomy. Exact matches score 1.0, parent-child matches
(e.g., Methodist vs Christian) score 0.7, and unrelated religions score 0.0.

This better reflects partial correctness: extracting "Christian Methodist" when
GT is "Methodist" is partially correct even if not exact.

### Helper Functions for Evaluation

```python
# Religion hierarchy: maps denominations to their parent category
RELIGION_HIERARCHY = {
    # Catholic tradition
    "catholic": "catholic",
    "roman catholic": "catholic",
    
    # Protestant denominations
    "methodist": "protestant",
    "united methodist": "protestant",
    "methodist church": "protestant",
    "baptist": "protestant",
    "southern baptist": "protestant",
    "presbyterian": "protestant",
    "episcopal": "protestant",
    "episcopalian": "protestant",
    "lutheran": "protestant",
    "evangelical": "protestant",
    "pentecostal": "protestant",
    "assemblies of god": "protestant",
    
    # Broadly Christian (covers generic "Christian" answers)
    "christian": "christian",
    "christian (unspecified)": "christian",
    "protestant": "protestant",
    
    # Other major religions
    "jewish": "jewish",
    "judaism": "jewish",
    "muslim": "muslim",
    "islam": "muslim",
    "jewish orthodox": "jewish",
    "orthodox": "orthodox",
    "mormon": "mormon",
    "church of jesus christ": "mormon",
    "lds": "mormon",
    "unitarian": "unitarian",
    "unitarian universalist": "unitarian",
    
    # Secular / None
    "atheist": "none",
    "agnostic": "none",
    "none": "none",
    "no religion": "none",
    "unaffiliated": "none",
}

def get_religion_category(religion_str):
    """
    Normalize religion string and return its category from the hierarchy.
    """
    if pd.isna(religion_str) or str(religion_str).strip() == "":
        return None
    
    norm = str(religion_str).strip().lower()
    
    # Direct lookup
    if norm in RELIGION_HIERARCHY:
        return RELIGION_HIERARCHY[norm]
    
    # Substring matching for common phrases
    for key, category in RELIGION_HIERARCHY.items():
        if key in norm or norm in key:
            return category
    
    # Default: treat as its own category
    return norm

def religion_match_score(gt_val, pred_val):
    """
    Hierarchical religion matching:
      - 1.0 for exact match (after lowercasing)
      - 0.7 for parent-child match (e.g., Methodist vs Christian)
      - 0.0 for unrelated religions
      - NaN when either GT or pred is missing (do not score as incorrect)
    
    This allows the model to get partial credit when extracting a parent
    category (Christian) when GT is a specific denomination (Methodist),
    reflecting the fact that it's partially correct.
    """
    if pd.isna(gt_val) or str(gt_val).strip() == "":
        return float("nan")
    if pd.isna(pred_val) or str(pred_val).strip() == "":
        return float("nan")
    
    gt_norm = str(gt_val).strip().lower()
    pred_norm = str(pred_val).strip().lower()
    
    # Exact match
    if gt_norm == pred_norm:
        return 1.0
    
    # Hierarchical match
    gt_cat = get_religion_category(gt_norm)
    pred_cat = get_religion_category(pred_norm)
    
    if gt_cat and pred_cat:
        if gt_cat == pred_cat:
            # Same category but different names —  partial credit
            return 0.7
    
    # No match
    return 0.0
```

**Scoring Details:**
- **1.0** — Exact match (Methodist = Methodist)
- **0.7** — Same religious category but different names (Methodist vs Christian, Baptist vs Protestant)
- **0.0** — Different religions (Methodist vs Catholic, Christian vs Muslim)
- **NaN** — Either value missing (not scored)

This approach rewards the model for extracting the correct parent category
when it can't pin down the exact denomination, which is semantically more
reasonable than treating all non-exact matches as completely wrong.